In [ ]:
pip install earthengine-api geemap

In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

# Define date range for 2025
start_date = '2025-01-01'
end_date = '2025-12-31'

# Define Brazil bounding box
bbox = ee.Geometry.BBox(-94.1875, -39.0208, 37.0625, 18.2292)

# Load the MOD16A2GF dataset
dataset = (
    ee.ImageCollection('MODIS/061/MOD16A2GF')
    .filterDate(start_date, end_date)
    .filterBounds(bbox)
)

# Select the Evapotranspiration (ET) band
evapotranspiration = dataset.select('ET')

print(f"Total ET images for 2025: {evapotranspiration.size().getInfo()}")
print(f"Bounding box: {bbox.getInfo()['coordinates']}")

In [ ]:
# Function to export each 8-day image
def export_image(image):
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
    task = ee.batch.Export.image.toDrive(
        image=image.clip(bbox),
        description=f"ET_Brazil_{date.getInfo()}",
        folder="Brazil_ET_8day",
        fileNamePrefix=f"ET_{date.getInfo()}",
        scale=500,  # MODIS native resolution
        region=bbox,
        maxPixels=1e13
    )
    task.start()
    print(f"Exporting: ET_{date.getInfo()}")

# Iterate through each image and export
et_list = evapotranspiration.toList(evapotranspiration.size())
for i in range(et_list.size().getInfo()):
    image = ee.Image(et_list.get(i))
    export_image(image)

print("\nAll ET export tasks initiated!")
print("Check Google Earth Engine Tasks tab for progress.")